# 02 · Road network

**Goal:** get real road distances and travel times between every pair of locations, correct the times for congestion, and add each stop's elevation.

### Load the stops
Reads the scenario from Notebook 1. `n` = number of locations (depot + stops).

In [1]:
import os

import numpy as np
import pandas as pd
import requests

CITY = os.environ.get("EMROUTE_CITY", "dhaka")
DATA_DIR = "../data"

stops = pd.read_csv(f"{DATA_DIR}/stops_{CITY}.csv")
n = len(stops)

print(f"{n} locations loaded (index 0 = depot)")
stops.head(3)

21 locations loaded (index 0 = depot)


,index,address,lat,lon,demand_kg,is_depot
0,0,"Tejgaon Industrial Area, Dhaka, Bangladesh",23.764878,90.405798,0.0,True
1,1,"Gulshan 1, Dhaka, Bangladesh",23.780553,90.416900,59.0,False
2,2,"Gulshan 2, Dhaka, Bangladesh",23.794719,90.413699,28.0,False


### Road distances and travel times (OSRM)
OSRM's `table` service returns an all-pairs matrix. Coordinates are passed as `lon,lat` (OSRM's order, not lat,lon), separated by `;`, and both `distance` (metres) and `duration` (seconds) are requested.

`distance_m[i, j]` and `duration_s[i, j]` are the road distance and driving time from location *i* to location *j*. Both are $n \times n$.

In [2]:
coords = ";".join(f"{row.lon},{row.lat}" for row in stops.itertuples())
url = f"https://router.project-osrm.org/table/v1/driving/{coords}"

response = requests.get(url, params={"annotations": "distance,duration"}, timeout=60)
response.raise_for_status()
result = response.json()

if result["code"] != "Ok":
    raise RuntimeError(f"OSRM returned: {result['code']} - {result.get('message')}")

distance_m = np.array(result["distances"])   # metres
duration_s = np.array(result["durations"])   # seconds

print(f"Matrix shape: {distance_m.shape}")
print(f"Depot -> {stops.loc[1, 'address'][:20]}: "
      f"{distance_m[0, 1]/1000:.2f} km, {duration_s[0, 1]/60:.1f} min")

Matrix shape: (21, 21)
Depot -> Gulshan 1, Dhaka, Ba: 3.56 km, 4.9 min


### Sanity checks
- **Zero diagonal:** travelling from a place to itself must cost 0. Anything else means rows and columns are misaligned.
- **No gaps:** a NaN means OSRM couldn't find a road route between two points (e.g. a coordinate that isn't near any road).
- **Asymmetry:** counts pairs where $d_{ij} \neq d_{ji}$. One-way streets and turn restrictions make A→B differ from B→A, which is why the full matrix is used rather than a symmetric one.

In [3]:
assert np.allclose(np.diag(distance_m), 0), "Diagonal is not zero -- matrix is misaligned!"
assert not np.isnan(distance_m).any(), "Some pairs are unroutable -- check your coordinates"

n_asym = (distance_m != distance_m.T).sum()
print(f"Asymmetric pairs: {n_asym} of {distance_m.size}")

Asymmetric pairs: 418 of 441


### Circuity factor
How much longer roads are than straight lines. Straight-line distances use the haversine formula (as in Notebook 1), computed for all pairs at once via broadcasting:

$$\text{circuity} = \underset{i \neq j}{\text{mean}}\ \frac{d^{\,\text{road}}_{ij}}{d^{\,\text{straight}}_{ij}}$$

A value of 1.48 means driving is 48% longer than as the crow flies. The diagonal is excluded because it is 0/0.

In [4]:
EARTH_RADIUS_M = 6_371_000.0

lat = np.radians(stops["lat"].values)
lon = np.radians(stops["lon"].values)

# Haversine for all pairs at once: column minus row gives an n x n matrix.
d_lat = lat[:, None] - lat[None, :]
d_lon = lon[:, None] - lon[None, :]
a = np.sin(d_lat / 2) ** 2 + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin(d_lon / 2) ** 2
straight_m = 2 * EARTH_RADIUS_M * np.arcsin(np.sqrt(a))

off_diagonal = ~np.eye(n, dtype=bool)   # leave out i == j (0 / 0)
circuity = (distance_m[off_diagonal] / straight_m[off_diagonal]).mean()

print(f"Mean circuity factor: {circuity:.3f}")
print(f"  -> driving in {CITY.title()} is {100*(circuity-1):.0f}% longer than the crow flies")

Mean circuity factor: 1.476
  -> driving in Dhaka is 48% longer than the crow flies


### Implied speed
Speed for each pair: $v = \dfrac{d}{t} \times 3.6$ (m/s → km/h). OSRM assumes free-flowing traffic, so these speeds are too high for Bangladeshi cities.

In [5]:
speeds = distance_m[off_diagonal] / duration_s[off_diagonal] * 3.6   # m/s -> km/h

print(f"OSRM implied speed: mean {speeds.mean():.1f} km/h "
      f"(range {speeds.min():.1f}-{speeds.max():.1f})")

OSRM implied speed: mean 47.3 km/h (range 19.6-69.5)


### Congestion correction
OSRM's times are stretched so the average speed matches a realistic target: 15 km/h for Dhaka, 25 km/h for Sylhet.

$$k = \frac{\bar v_{\text{OSRM}}}{v_{\text{target}}}, \qquad t'_{ij} = k \cdot t_{ij}$$

Distances are unchanged; only durations change. This matters because fuel per km depends on speed (Notebook 3).

In [6]:
TARGET_SPEED_KMH = {
    "dhaka":  15.0,   # severe, chronic congestion
    "sylhet": 25.0,   # smaller city, lighter traffic
}[CITY]

congestion_factor = speeds.mean() / TARGET_SPEED_KMH
duration_s = duration_s * congestion_factor

corrected_speeds = distance_m[off_diagonal] / duration_s[off_diagonal] * 3.6

print(f"Applied congestion factor: {congestion_factor:.2f}x slower")
print(f"Corrected mean speed: {corrected_speeds.mean():.1f} km/h")

Applied congestion factor: 3.15x slower
Corrected mean speed: 15.0 km/h


### Elevation
Each stop's elevation comes from the SRTM 30 m dataset via OpenTopoData. The **spread** (max − min) shows how much terrain the gradient term has to work with.

SRTM's vertical error is several metres or more, so a spread under ~30 m is mostly noise. Dhaka falls in this case, which is why Sylhet is also studied.

In [7]:
locations = "|".join(f"{row.lat},{row.lon}" for row in stops.itertuples())

response = requests.get("https://api.opentopodata.org/v1/srtm30m",
                        params={"locations": locations}, timeout=60)
response.raise_for_status()
elevation_data = response.json()

if elevation_data["status"] != "OK":
    raise RuntimeError(f"Elevation API: {elevation_data}")

stops["elevation_m"] = [point["elevation"] for point in elevation_data["results"]]

spread = stops["elevation_m"].max() - stops["elevation_m"].min()
print(f"Elevation range: {stops['elevation_m'].min():.0f} to "
      f"{stops['elevation_m'].max():.0f} m  (spread {spread:.0f} m)")

if spread < 30:
    print("\n[!] Terrain spread is within SRTM error -- gradient term unresolvable here.")
    print("    Report this explicitly. It is the reason Sylhet is in the study.")

Elevation range: 5 to 24 m  (spread 19 m)

[!] Terrain spread is within SRTM error -- gradient term unresolvable here.
    Report this explicitly. It is the reason Sylhet is in the study.


### Save
Distance matrix, congestion-corrected duration matrix, stops (now with `elevation_m`), and a one-row network summary.

In [8]:
pd.DataFrame(distance_m).to_csv(f"{DATA_DIR}/distance_m_{CITY}.csv", index=False)
pd.DataFrame(duration_s).to_csv(f"{DATA_DIR}/duration_s_{CITY}.csv", index=False)
stops.to_csv(f"{DATA_DIR}/stops_{CITY}.csv", index=False)   # now includes elevation

print("Saved:")
print(f"  {DATA_DIR}/distance_m_{CITY}.csv   ({n}x{n} metres)")
print(f"  {DATA_DIR}/duration_s_{CITY}.csv   ({n}x{n} seconds, congestion-corrected)")
print(f"  {DATA_DIR}/stops_{CITY}.csv        (+ elevation_m column)")

summary = {
    "city": CITY,
    "locations": n,
    "circuity_factor": round(circuity, 3),
    "mean_speed_kmh": round(corrected_speeds.mean(), 1),
    "elevation_spread_m": float(spread),
    "asymmetric_pairs": int(n_asym),
}
pd.DataFrame([summary]).to_csv(f"{DATA_DIR}/network_summary_{CITY}.csv", index=False)
summary

Saved:
  ../data/distance_m_dhaka.csv   (21x21 metres)
  ../data/duration_s_dhaka.csv   (21x21 seconds, congestion-corrected)
  ../data/stops_dhaka.csv        (+ elevation_m column)


{'city': 'dhaka',
 'locations': 21,
 'circuity_factor': np.float64(1.476),
 'mean_speed_kmh': np.float64(15.0),
 'elevation_spread_m': 19.0,
 'asymmetric_pairs': 418}

## Conclusions

| | Dhaka | Sylhet |
|---|---|---|
| Circuity (road ÷ straight line) | 1.476 | 1.535 |
| Asymmetric pairs | 418 / 441 | 234 / 272 |
| Assumed traffic speed | 15 km/h | 25 km/h |
| Elevation spread | 19 m | 45 m |

- **Straight-line distance would be badly wrong:** real roads are ~50% longer in both cities.
- **The problem is asymmetric:** almost every pair has A→B ≠ B→A because of one-way streets, so the solver must use the full matrix.
- **OSRM's speeds are free-flow fantasy** (~47 km/h mean in Dhaka). Stretching durations by ~3.1× brings Dhaka to a realistic 15 km/h. This matters because fuel per km depends on speed.
- **Dhaka is effectively flat:** its 19 m spread is within SRTM's error, so its gradient term should be read as zero. Sylhet's 45 m is the only real relief in the study, and it is still modest.